In [ ]:
#| default_exp cli

In [ ]:
#| export
"""Command-line interface for healpyxel."""

import sys

def sidecar_cli():
    """CLI entry point for healpix_sidecar command."""
    from healpyxel import sidecar
    return sidecar.main()

def aggregate_cli():
    """CLI entry point for healpix_aggregate command."""
    from healpyxel import aggregate
    return aggregate.main()

def accumulator_cli():
    """CLI entry point for healpix_accumulator command."""
    from healpyxel import accumulator
    return accumulator.main()

def finalize_cli():
    """CLI entry point for healpix_finalize command."""
    from healpyxel import finalize
    return finalize.main()

def to_geoparquet_cli():
    """CLI entry point for healpyxel_to_geoparquet command."""
    from healpyxel import geospatial
    return geospatial.main()

In [ ]:
#| export
def cache_cli():
    """CLI entry point for healpyxel-cache command."""
    import click
    import os
    from healpyxel.geospatial import manage_healpix_cache
    
    @click.command('healpyxel-cache', context_settings={'help_option_names': ['-h', '--help']})
    @click.option('--list', 'action_list', is_flag=True, help='List cached grids')
    @click.option('--generate', type=int, multiple=True, help='Generate cache for specific nsides (e.g., --generate 32 --generate 256)')
    @click.option('--verify', type=int, multiple=True, help='Verify cache integrity for specific nsides (e.g., --verify 256 --verify 512)')
    @click.option('--clean', is_flag=True, help='Remove all cache files')
    @click.option('--info', is_flag=True, help='Show cache directory info')
    @click.option('--config', is_flag=True, help='Show configuration and precedence')
    @click.option('--cache-dir', type=click.Path(), default=None,
                  help='Override cache directory. Precedence: CLI > HEALPYXEL_CACHE > $XDG_CACHE_HOME/healpyxel/healpix_grids > $HOME/.cache/healpyxel/healpix_grids')
    @click.option('--config-dir', type=click.Path(), default=None,
                  help='Override config directory. Precedence: CLI > HEALPYXEL_CONFIG > $XDG_CONFIG_HOME/healpyxel > $HOME/.config/healpyxel')
    @click.option('--force', is_flag=True, help='Overwrite existing cache files during --generate')
    @click.option('--quiet', is_flag=True, help='Minimal output (JSON)')
    def cmd(action_list, generate, verify, clean, info, config, cache_dir, config_dir, force, quiet):
        """Manage HEALPix grid cache with XDG compliance.
        
        Examples:
            # List cached grids (uses default XDG locations)
            healpyxel-cache --list
            
            # Show current configuration and how precedence resolves
            healpyxel-cache --config
            
            # Generate cache for specific nsides
            healpyxel-cache --generate 32 --generate 256
            
            # Verify cache integrity (recommended for CI/production)
            healpyxel-cache --verify 256 --verify 512
            
            # Override cache directory for this command only
            healpyxel-cache --cache-dir /fast_ssd/healpyxel --generate 512
            
            # Use environment variable for all healpyxel commands
            export HEALPYXEL_CACHE=/mnt/ssd/healpix_grids
            healpyxel-cache --list
            
            # Respect XDG spec (use custom cache home)
            export XDG_CACHE_HOME=/data/cache
            healpyxel-cache --list  # Uses /data/cache/healpyxel/healpix_grids
        """
        import json
        from pathlib import Path
        
        # Determine action
        if action_list:
            action = 'list'
        elif generate:
            action = 'generate'
        elif verify:
            action = 'verify'
        elif clean:
            action = 'clean'
        elif info:
            action = 'info'
        elif config:
            action = 'config'
        else:
            click.echo('No action specified. Use --list, --generate, --verify, --clean, --info, or --config.')
            click.echo('Run with -h for help and precedence details.')
            return
        
        # Dispatch to domain logic
        try:
            result = manage_healpix_cache(
                action=action,
                nsides=list(generate) if generate else (list(verify) if verify else None),
                cache_dir=Path(cache_dir) if cache_dir else None,
                config_dir=Path(config_dir) if config_dir else None,
                force=force
            )
        except Exception as e:
            click.echo(f'❌ Error: {e}', err=True)
            raise click.Abort()
        
        # Format output
        if quiet:
            click.echo(json.dumps(result, default=str, indent=2))
        else:
            if action == 'list':
                click.echo(f"Cache directory: {result['cache_dir']}")
                if not result['files']:
                    click.echo('No cached grids found.')
                else:
                    click.echo(f"Cached grids ({result['count']}):")
                    for f in result['files']:
                        click.echo(f"  {f['filename']:45s} {f['cells']:6d} cells  {f['size_mb']:7.1f} MB")
            
            elif action == 'verify':
                # Report verification results with appropriate exit code
                has_errors = result['status'] == 'error'
                for ver in result['verified']:
                    if ver['status'] == 'ok':
                        click.echo(f"✓ nside={ver['nside']:3d}  OK ({ver['cells']:6d} cells, {ver['size_mb']:7.1f} MB)")
                    elif ver['status'] == 'missing':
                        click.echo(f"✗ nside={ver['nside']:3d}  MISSING: {ver['error']}", err=True)
                    elif ver['status'] == 'incomplete':
                        click.echo(f"✗ nside={ver['nside']:3d}  INCOMPLETE: {ver['error']} ({ver.get('missing_count', 0)} missing)", err=True)
                    elif ver['status'] == 'corrupt':
                        click.echo(f"✗ nside={ver['nside']:3d}  CORRUPT: {ver['error']}", err=True)
                    else:
                        click.echo(f"✗ nside={ver['nside']:3d}  ERROR: {ver.get('error', 'unknown error')}", err=True)
                
                if has_errors:
                    raise click.Abort()  # Non-zero exit code for CI/CD
            
            elif action == 'config':
                click.echo(f"Config file: {result['config_file']}")
                click.echo(f"Exists: {result['config_exists']}")
                click.echo()
                click.echo('Current Settings:')
                for key, val in result['settings'].items():
                    click.echo(f"  {key:25s} {val}")
                click.echo()
                click.echo('Precedence Resolution:')
                for key, val in result['precedence'].items():
                    click.echo(f"  {key:25s} {val}")
            
            elif action == 'generate':
                for gen in result['generated']:
                    status_icon = '✓' if gen['status'] == 'ok' else ('⊘' if gen['status'] == 'skipped' else '✗')
                    msg = f"{status_icon} nside={gen['nside']:3d}"
                    if gen['status'] == 'ok':
                        msg += f"  {gen['cells']:6d} cells"
                    elif gen['status'] == 'skipped':
                        msg += f"  {gen['reason']}"
                    else:
                        msg += f"  ERROR: {gen['error']}"
                    click.echo(msg)
            
            elif action == 'clean':
                if result['deleted'] > 0:
                    click.echo(f"✓ Deleted {result['deleted']} cache file(s) from {result['cache_dir']}")
                else:
                    click.echo('No cache files to delete.')
            
            elif action == 'info':
                click.echo(f"Cache directory: {result['cache_dir']}")
                click.echo(f"  Exists: {result['cache_dir_exists']}")
                click.echo(f"  Files: {result['total_files']}")
                click.echo(f"  Total size: {result['total_size_mb']:.1f} MB")
                click.echo()
                click.echo(f"Config directory: {result['config_dir']}")
                click.echo(f"  Exists: {result['config_dir_exists']}")
    
    return cmd()

## CLI Entry Points

These functions are called when using the command-line tools:
- `healpix_sidecar` → `sidecar_cli()`
- `healpix_aggregate` → `aggregate_cli()`
- `healpix_accumulator` → `accumulator_cli()`
- `healpix_finalize` → `finalize_cli()`
- `healpyxel_to_geoparquet` → `to_geoparquet_cli()`
- `healpyxel-cache` → `cache_cli()`


## Usage

After installing the package, these commands will be available:

```bash
# Generate HEALPix sidecar
healpix_sidecar --input data.parquet --nside 64 --mode fuzzy

# Batch aggregation
healpix_aggregate --input data.parquet --sidecar data.sidecar.parquet --columns r750 r950

# Streaming accumulation
healpix_accumulator --input batch001.parquet --sidecar batch001.sidecar.parquet --columns r750

# Finalize to maps
healpix_finalize --state state.parquet --output final_map.parquet --min-count 5

# Manage HEALPix grid cache
healpyxel-cache --list                          # List cached grids
healpyxel-cache --generate 32 --generate 256    # Generate cache for nsides
healpyxel-cache --verify 256 --verify 512       # Verify cache integrity (CI/production)
healpyxel-cache --config                        # Show configuration + precedence
healpyxel-cache --clean                         # Remove all cache files
healpyxel-cache --info                          # Show cache statistics
healpyxel-cache --cache-dir /tmp --generate 64  # Override cache location
```